# Taller 5 – Punto 3: Dataset de Kaggle y modelo neuronal

Se utiliza el dataset **Breast Cancer Wisconsin (Diagnostic)** disponible en Kaggle. El objetivo es clasificar cada observación como tumor **benigno (B)** o **maligno (M)** a partir de características numéricas obtenidas de imágenes digitalizadas de núcleos celulares.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Descargar el dataset desde Kaggle
ruta = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print("Ruta:", ruta)

archivos_csv = glob.glob(os.path.join(ruta, "*.csv"))
print("CSV encontrados:", archivos_csv)

df = pd.read_csv(archivos_csv[0])
df.head()

In [ ]:
print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

print("\nDistribución del rótulo diagnosis:")
print(df["diagnosis"].value_counts())

print("\nValores faltantes:")
print(df.isna().sum().sort_values(ascending=False).head())

## Features y rótulo

El rótulo es `diagnosis`:

- `M`: maligno.
- `B`: benigno.

Las variables de entrada son 30 características numéricas. Se derivan de diez propiedades básicas del núcleo celular —radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal— y se expresan mediante tres grupos: **media**, **error estándar** y **peor valor**.

In [ ]:
# Limpieza
df = df.drop(columns=["id"], errors="ignore")
df = df.drop(columns=["Unnamed: 32"], errors="ignore")

# Convertir el rótulo a binario: M=1, B=0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Número de features:", X.shape[1])
print("Número de ejemplos:", X.shape[0])

In [ ]:
# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# Estandarización
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Entrenamiento:", X_train_s.shape)
print("Prueba:", X_test_s.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

hist = model.fit(
    X_train_s,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
resultados = model.evaluate(X_test_s, y_test, verbose=0)
metricas = dict(zip(model.metrics_names, resultados))
print("Resultados de prueba:", metricas)

prob = model.predict(X_test_s, verbose=0).ravel()
pred = (prob >= 0.5).astype(int)

print("\nReporte de clasificación:")
print(classification_report(y_test, pred, target_names=["Benigno", "Maligno"]))

cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm, display_labels=["Benigno", "Maligno"]).plot()
plt.title("Matriz de confusión")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hist.history["loss"], label="Entrenamiento")
plt.plot(hist.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Evolución de la pérdida")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(hist.history["accuracy"], label="Entrenamiento")
plt.plot(hist.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Evolución de la exactitud")
plt.legend()
plt.grid(True)
plt.show()

## Qué analizar en el informe

1. Comparar la pérdida de entrenamiento y validación para identificar si existe sobreentrenamiento.
2. Reportar `accuracy`, `precision`, `recall` y `AUC`.
3. Revisar especialmente los **falsos negativos** de la matriz de confusión, porque corresponden a casos malignos clasificados como benignos.
4. Explicar que la estandarización evita que variables con escalas numéricas grandes dominen el aprendizaje.
5. Aclarar que el modelo es académico y no constituye una herramienta clínica.